In [4]:
import os

print(os.getcwd())
print(os.listdir())



C:\Users\SATYA\OneDrive\Desktop\voice-stress-detection-project
['.idea', '.ipynb_checkpoints', '.venv', 'archive', 'data', 'models', 'requirements.txt', 'src', 'Untitled.ipynb']


In [5]:
print(os.path.exists("archive"))

True


In [6]:
print(os.listdir("archive")[:10])

['.idea', 'Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05', 'Actor_06', 'Actor_07', 'Actor_08', 'Actor_09']


In [7]:
import os
import pandas as pd

DATA_DIR = "archive"

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

label_map = {
    "neutral": "calm",
    "calm": "calm",
    "angry": "stressed",
    "fearful": "stressed"
}

rows = []

for actor_folder in os.listdir(DATA_DIR):
    actor_path = os.path.join(DATA_DIR, actor_folder)

    if not os.path.isdir(actor_path):
        continue

    for file in os.listdir(actor_path):
        if not file.endswith(".wav"):
            continue

        parts = file.split("-")
        emotion_code = parts[2]
        emotion = emotion_map.get(emotion_code)

        if emotion in label_map:
            rows.append({
                "filepath": os.path.join(actor_path, file),
                "emotion": emotion,
                "label": label_map[emotion]
            })

df = pd.DataFrame(rows)

df.to_csv("data/labels.csv", index=False)

print("Total files labeled:", len(df))
print(df["label"].value_counts())

Total files labeled: 672
label
stressed    384
calm        288
Name: count, dtype: int64


In [1]:
%pip install tqdm

  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

print(os.getcwd())

C:\Users\SATYA\OneDrive\Desktop\voice-stress-detection-project


In [4]:
import os

print(os.path.exists("../data/labels.csv"))
print(os.path.exists("data/labels.csv"))

False
True


In [6]:
import pandas as pd
import numpy as np
import librosa
import os
from tqdm import tqdm

df = pd.read_csv("data/labels.csv")

MAX_LEN = 130   # fixed number of time frames per clip (for LSTM input)
N_MFCC = 13

def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=22050)
    y, _ = librosa.effects.trim(y)  # remove silence

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    pitch, _ = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitch[pitch > 0]) if np.any(pitch > 0) else 0
    energy = librosa.feature.rms(y=y)

    # Pad or truncate MFCC to fixed length for LSTM input
    if mfcc.shape[1] < MAX_LEN:
        pad_width = MAX_LEN - mfcc.shape[1]
        mfcc_padded = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')
    else:
        mfcc_padded = mfcc[:, :MAX_LEN]

    # Aggregated (statistical) features for baseline model
    agg_features = np.concatenate([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        [pitch_mean],
        [np.mean(energy)], [np.std(energy)]
    ])

    return mfcc_padded.T, agg_features  # sequence (for LSTM), aggregated (for baseline)

sequence_data = []
aggregated_data = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        seq, agg = extract_features(row["filepath"])
        sequence_data.append(seq)
        aggregated_data.append(agg)
        labels.append(row["label"])
    except Exception as e:
        print("Skipped:", row["filepath"], e)

X_seq = np.array(sequence_data)
X_agg = np.array(aggregated_data)
y = np.array(labels)

np.save("data/X_seq.npy", X_seq)
np.save("data/X_agg.npy", X_agg)
np.save("data/y.npy", y)

print("Sequence data shape:", X_seq.shape)
print("Aggregated data shape:", X_agg.shape)
print("Labels shape:", y.shape)
print("Features saved successfully!")

100%|██████████| 672/672 [00:34<00:00, 19.53it/s]


Sequence data shape: (672, 130, 13)
Aggregated data shape: (672, 29)
Labels shape: (672,)
Features saved successfully!


In [12]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\SATYA\OneDrive\Desktop\voice-stress-detection-project
['.idea', '.ipynb_checkpoints', '.venv', 'archive', 'data', 'models', 'requirements.txt', 'src', 'Untitled.ipynb']


In [14]:
import os

for root, dirs, files in os.walk("."):
    if "extract_features.py" in files:
        print(os.path.join(root, "extract_features.py"))

In [16]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\SATYA\OneDrive\Desktop\voice-stress-detection-project
['.idea', '.ipynb_checkpoints', '.venv', 'archive', 'data', 'extract_features.ipynb', 'models', 'requirements.txt', 'src', 'Untitled.ipynb']


In [17]:
import os

print(os.listdir("data"))

['labels.csv', 'X_agg.npy', 'X_seq.npy', 'y.npy']
